# 09 — Les listes de noms et prénoms de l'Insee peuvent-elles aider la lecture ?

**Notebook autonome** : il télécharge lui-même les listes de l'Insee, charge les bulletins et le
Fiqual depuis S3, et appelle lui-même gemma4. Il ne dépend d'aucun autre notebook ni module.


In [5]:
# 1. Reglages et outils (tout est ici)
import base64
import io
import json
import math
import os
import re
import shutil
import subprocess
import time
import unicodedata
import zipfile
from pathlib import Path

import mlflow
import pandas as pd
import requests
from PIL import Image
from scipy import stats

TEST_RAPIDE = False             # True : 10 bulletins (verification). False : tous (169 pour RG25).
MODELE = "gemma4-26b-moe"
LONGUEUR_MIN_CORRECTION = 4      # on ne corrige pas les mots trop courts (trop d'homonymes)

BUCKET, CAMPAGNE = "projet-mesure-qualite-rp", "RG25"
FICHIER_FIQUAL = "RP_PMQ_FIQUAL2599.txt"
POS_UT, POS_CAB = 2, 4
COLS = {"NOM": 11, "PRENOM": 12}
BORDS_BLOC = (0.075, 0.072, 0.450, 0.105)
g, h, d_, b_ = BORDS_BLOC
ZONE_BLOC = (g, h, d_ - g, b_ - h)

URL_NOMS = "https://www.insee.fr/fr/statistiques/fichier/3536630/noms2008nat_txt.zip"
URL_PRENOMS = "https://www.insee.fr/fr/statistiques/fichier/8595130/prenoms-2025-nat_csv.zip"

RESULTATS = Path.home() / "work" / "banc-htr" / "resultats"
LISTES = Path.home() / "work" / "banc-htr" / "listes_insee"   # petits fichiers, gardes
BLOCS = Path("/tmp/listes-insee/blocs")                       # images, refaites si besoin
for d in [RESULTATS, LISTES, BLOCS]:
    d.mkdir(parents=True, exist_ok=True)
SUFFIXE = "_rapide" if TEST_RAPIDE else ""


def normaliser(t):
    """Identique aux notebooks precedents : majuscules, sans accent, ponctuation -> espace."""
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return ""
    t = unicodedata.normalize("NFKD", str(t).strip())
    t = "".join(c for c in t if not unicodedata.combining(c)).upper()
    t = re.sub(r"[^A-Z0-9 ]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def distance(a, b):
    if a == b:
        return 0
    prec = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cour = [i]
        for j, cb in enumerate(b, 1):
            cour.append(min(prec[j] + 1, cour[j - 1] + 1, prec[j - 1] + (ca != cb)))
        prec = cour
    return prec[-1]


def wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, c - m), min(1.0, c + m)


def mcnemar(avant, apres):
    """Test exact entre deux series de 0/1 sur les memes lignes : (reparees, abimees, p)."""
    rep = int(((avant == 0) & (apres == 1)).sum())
    abi = int(((avant == 1) & (apres == 0)).sum())
    p = stats.binomtest(rep, rep + abi).pvalue if rep + abi else 1.0
    return rep, abi, p


print(f"mode {'TEST RAPIDE' if TEST_RAPIDE else 'COMPLET'} | modele {MODELE}")

mode COMPLET | modele gemma4-26b-moe


In [6]:
# 2. Les listes de l'Insee : telechargees une fois, gardees dans ~/work/banc-htr/listes_insee
def telecharger(url, dossier):
    f = dossier / url.split("/")[-1]
    if not f.exists():
        print(f"telechargement de {f.name}...", flush=True)
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        f.write_bytes(r.content)
    return f


def lire_zip(f):
    with zipfile.ZipFile(f) as z:
        membre = next(n for n in z.namelist() if n.lower().endswith((".txt", ".csv")))
        brut = z.read(membre)
    for enc in ("utf-8", "latin-1"):
        try:
            texte = brut.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    premiere = texte.split("\n", 1)[0]
    sep = max(["\t", ";", ","], key=premiere.count)
    return pd.read_csv(io.StringIO(texte), sep=sep, dtype=str, keep_default_na=False)


try:
    noms_brut = lire_zip(telecharger(URL_NOMS, LISTES))
    prenoms_brut = lire_zip(telecharger(URL_PRENOMS, LISTES))
except Exception as e:
    raise RuntimeError(
        f"Telechargement impossible ({type(e).__name__}). Telecharger a la main les deux fichiers :\n"
        f"  {URL_NOMS}\n  {URL_PRENOMS}\n"
        f"puis les deposer, sans les dezipper, dans {LISTES} et relancer cette cellule.")

# Noms : premiere colonne = nom ; les autres = effectifs par decennie
noms_brut.columns = [c.strip().upper() for c in noms_brut.columns]
col_nom = noms_brut.columns[0]
effectifs = noms_brut.drop(columns=[col_nom]).apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
NOMS = (pd.DataFrame({"nom": noms_brut[col_nom].map(normaliser), "n": effectifs})
        .query("nom != '' and nom != 'AUTRES NOMS'").groupby("nom").n.sum().to_dict())

# Prenoms : colonnes SEXE, PRENOM, PERIODE, VALEUR (noms anciens acceptes : PREUSUEL, NOMBRE)
prenoms_brut.columns = [c.strip().upper() for c in prenoms_brut.columns]
col_p = next(c for c in ["PRENOM", "PREUSUEL"] if c in prenoms_brut.columns)
col_v = next(c for c in ["VALEUR", "NOMBRE"] if c in prenoms_brut.columns)
p_ = pd.DataFrame({"prenom": prenoms_brut[col_p].map(normaliser),
                   "n": pd.to_numeric(prenoms_brut[col_v], errors="coerce").fillna(0)})
p_ = p_[(p_.prenom != "") & ~p_.prenom.str.contains("PRENOMS RARES")]
PRENOMS = p_.groupby("prenom").n.sum().to_dict()
LISTE = {"NOM": NOMS, "PRENOM": PRENOMS}
print(f"{len(NOMS):,} noms et {len(PRENOMS):,} prenoms dans les listes de l'Insee".replace(",", " "))

218 114 noms et 43 449 prenoms dans les listes de l'Insee


In [7]:
# 3. Les bulletins de RG25, le Fiqual et les images des blocs (charges ici, depuis S3)
import s3fs
fs = s3fs.S3FileSystem()
fichiers = [f for f in fs.find(f"{BUCKET}/data/raw/{CAMPAGNE}/BIE") if f.endswith("_BI.jpg")]
bi = pd.DataFrame({"chemin": fichiers})
m = bi.chemin.map(lambda c: Path(c).stem).str.split("_")
bi["cle"] = m.str[0] + "_" + m.str[1]
with fs.open(f"{BUCKET}/data/raw/{CAMPAGNE}/{FICHIER_FIQUAL}", "rb") as f:
    brut = pd.read_csv(f, sep="\x00", header=None, names=["ligne"], dtype=str,
                       encoding="latin-1", keep_default_na=False, engine="python")["ligne"]
ch = brut.str.split("|")
fq3 = pd.DataFrame(ch[ch.str[0] == "3"].tolist(), dtype=str)
fq3["cle"] = fq3[POS_UT].str.strip() + "_" + fq3[POS_CAB].str.strip()
ref = fq3[["cle", COLS["NOM"], COLS["PRENOM"]]].copy()
ref.columns = ["cle", "NOM", "PRENOM"]
APP = bi.merge(ref, on="cle", how="inner").drop_duplicates("cle").sort_values("cle").reset_index(drop=True)
APP = APP.head(10) if TEST_RAPIDE else APP
REF = {(r.cle, c): normaliser(getattr(r, c)) for r in APP.itertuples() for c in COLS}

for r in APP.itertuples():
    fb = BLOCS / f"{r.cle}.png"
    if not fb.exists():
        with fs.open(r.chemin, "rb") as src:
            page = Image.open(io.BytesIO(src.read())).convert("RGB")
        L, H = page.size
        x, y, w, hh = ZONE_BLOC
        page.crop((int(x * L), int(y * H), int((x + w) * L), int((y + hh) * H))).save(fb)
print(f"{len(APP)} bulletins, {len(REF)} champs")

169 bulletins, 338 champs


In [8]:
# 4. Le lecteur gemma4 (meme consigne que les notebooks 05 et 07)
import numpy as np
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

N_PARALLELE = 4          # appels simultanes au service ; baisser a 2 en cas d'erreurs 429

BASE_URL = "https://llm.lab.sspcloud.fr/api"
if "OPENAI_API_KEY" not in os.environ and shutil.which("vault"):
    r = subprocess.run(["vault", "kv", "get", "-field=OPENAI_API_KEY", "onyxia-kv/farisazibert/LLM"],
                       capture_output=True, text=True, timeout=60)
    if r.returncode == 0 and r.stdout.strip():
        os.environ["OPENAI_API_KEY"] = r.stdout.strip()
if "OPENAI_API_KEY" not in os.environ:
    raise RuntimeError("Cle du service absente : la renseigner (Vault) avant de continuer.")
HEADERS = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}

# --- Journal : un fichier detaille (erreurs, temps, progression), sans aucune lecture (donc sans nom) ---
JOURNAL = RESULTATS / f"journal_listes_insee{SUFFIXE}.log"
F_PROGRESSION = RESULTATS / f"progression_lecture{SUFFIXE}.csv"
log = logging.getLogger("listes_insee")
log.setLevel(logging.INFO)
if not log.handlers:
    h = logging.FileHandler(JOURNAL, encoding="utf-8")
    h.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    log.addHandler(h)

PROMPT_BLOC = """Cette image montre un extrait de bulletin individuel du recensement francais.
Elle contient des lignes manuscrites precedees de leurs libelles imprimes :
"Nom :" et "Prenom :".

Transcris ce qui est ecrit a la main sur chacune, en MAJUSCULES, sans accent.

Regles :
- Ne corrige rien, n'ajoute rien, ne devine pas.
- Ne recopie pas les libelles imprimes, seulement l'ecriture manuscrite.
- Si une ligne n'est pas lisible avec certitude, mets exactement ILLISIBLE.

Reponds uniquement en JSON : {"NOM": "...", "PRENOM": "..."}"""

PROMPT_RELECTURE = """Cette image montre un extrait de bulletin individuel du recensement francais,
avec un nom et un prenom ecrits a la main apres les libelles imprimes "Nom :" et "Prenom :".

Une premiere lecture a donne : {lecture}
Ces valeurs ne figurent pas dans les listes de l'Insee des noms et prenoms portes en France.
Voici des valeurs existantes, proches de la premiere lecture :
{candidats}

Relis attentivement l'ecriture manuscrite.
- Si elle correspond vraiment a l'une de ces valeurs, reponds cette valeur.
- Sinon, recopie exactement ce qui est ecrit, meme si ce n'est pas dans la liste.
- Si ce n'est pas lisible avec certitude, mets exactement ILLISIBLE.

Reponds uniquement en JSON : {{"NOM": "...", "PRENOM": "..."}}"""


def _b64(image):
    t = io.BytesIO()
    image.convert("RGB").save(t, format="PNG")
    return base64.b64encode(t.getvalue()).decode()


def lire(image, consigne, tentatives=3, etiquette=""):
    """Renvoie ({champ: texte}, reussi). En cas d'echec, reussi = False (a ne pas mettre en cache)."""
    for essai in range(tentatives):
        t0 = time.time()
        try:
            r = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS, timeout=180, json={
                "model": MODELE, "temperature": 0, "max_tokens": 200,
                "messages": [{"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{_b64(image)}"}},
                    {"type": "text", "text": consigne}]}]})
            if r.status_code == 429 or r.status_code >= 500:
                raise RuntimeError(f"HTTP {r.status_code}")
            r.raise_for_status()
            texte = r.json()["choices"][0]["message"]["content"]
            m = re.search(r"\{.*\}", texte or "", re.S)
            d = json.loads(m.group(0)) if m else {}
            log.info(f"{etiquette} ok {time.time() - t0:.1f}s essai={essai + 1} json={'oui' if m else 'non'}")
            return {c: str(d.get(c, "") or "") for c in COLS}, True
        except Exception as e:
            log.warning(f"{etiquette} echec essai={essai + 1} {type(e).__name__}: {str(e)[:120]}")
            time.sleep(2 * (essai + 1))
    log.error(f"{etiquette} abandon apres {tentatives} essais")
    return {c: "" for c in COLS}, False


def lire_bloc(cle):
    t0 = time.time()
    res, ok = lire(Image.open(BLOCS / f"{cle}.png"), PROMPT_BLOC, etiquette=f"bloc={cle}")
    return cle, res, ok, time.time() - t0


# --- Premiere lecture de tous les blocs (cache ; une relance reprend et retente les echecs) ---
cache = RESULTATS / f"listes_insee_base{SUFFIXE}.csv"
BASE = pd.read_csv(cache, dtype=str).fillna("") if cache.exists() else pd.DataFrame(columns=["cle", "champ", "lu"])
a_faire = [c for c in APP.cle if c not in set(BASE.cle)]
SESSION = datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"{BASE.cle.nunique()} blocs deja lus, {len(a_faire)} a lire | journal : {JOURNAL}")
log.info(f"session {SESSION} : {len(a_faire)} blocs a lire, {N_PARALLELE} en parallele, modele {MODELE}")

rows, echecs, temps = [], [], []
PROGRESSION = []                                 # une ligne par bloc lu : l'evolution de la lecture
t_debut, dernier_affichage = time.time(), 0.0


def sauver():
    """Ecrit les lectures reussies dans le cache, et l'evolution dans le fichier de progression."""
    global BASE, rows, PROGRESSION
    if rows:
        BASE = pd.concat([BASE, pd.DataFrame(rows)], ignore_index=True)
        BASE.to_csv(cache, index=False)
        rows = []
    if PROGRESSION:
        pd.DataFrame(PROGRESSION).to_csv(F_PROGRESSION, mode="a", index=False,
                                         header=not F_PROGRESSION.exists())
        PROGRESSION_SESSION.extend(PROGRESSION)
        PROGRESSION = []


PROGRESSION_SESSION = []                         # toute l'evolution de cette session (pour MLflow)

try:
    with ThreadPoolExecutor(N_PARALLELE) as ex:
        futures = [ex.submit(lire_bloc, c) for c in a_faire]
        for k, f in enumerate(as_completed(futures), 1):
            cle, res, ok, s = f.result()
            temps.append(s)
            if ok:
                rows += [{"cle": cle, "champ": c, "lu": res[c]} for c in COLS]
            else:
                echecs.append(cle)
            ecoule = time.time() - t_debut
            PROGRESSION.append({"session": SESSION, "bloc": k, "ecoule_s": round(ecoule, 1),
                                "s_appel": round(s, 2), "debit_par_min": round(k / ecoule * 60, 1),
                                "echecs_cumules": len(echecs), "reussi": int(ok)})
            if k % 20 == 0 or k == len(a_faire):
                sauver()
            if time.time() - dernier_affichage > 30 or k == len(a_faire):     # toutes les 30 s
                dernier_affichage = time.time()
                msg = (f"{k}/{len(a_faire)} | {ecoule / 60:.1f} min | {np.mean(temps[-20:]):.1f} s/appel (20 derniers) | "
                       f"{k / ecoule * 60:.0f} blocs/min | reste ~{ecoule / k * (len(a_faire) - k) / 60:.0f} min | "
                       f"echecs {len(echecs)}")
                print("  " + msg, flush=True)
                log.info("progression " + msg)
finally:
    sauver()                                     # interruption : rien n'est perdu

STATS_LECTURE = {"blocs_lus": int(BASE.cle.nunique()), "appels": len(temps), "echecs": len(echecs),
                 "s_par_appel": float(np.mean(temps)) if temps else 0.0,
                 "duree_min": (time.time() - t_debut) / 60, "parallele": N_PARALLELE}
log.info(f"fin de session {SESSION} : {STATS_LECTURE}")
print(f"premiere lecture : {BASE.cle.nunique()} blocs | {len(echecs)} echecs"
      + (" (relancer la cellule pour les retenter)" if echecs else "")
      + f" | evolution : {F_PROGRESSION}")

169 blocs deja lus, 0 a lire | journal : /home/onyxia/work/banc-htr/resultats/journal_listes_insee.log
premiere lecture : 169 blocs | 0 echecs | evolution : /home/onyxia/work/banc-htr/resultats/progression_lecture.csv


In [9]:
import time, requests

t0 = time.time()
r = requests.get(f"{BASE_URL}/models", headers=HEADERS, timeout=30)
print(f"liste des modeles : HTTP {r.status_code} en {time.time() - t0:.1f} s")
disponibles = [m["id"] for m in r.json().get("data", [])] if r.ok else []
print(f"{MODELE} disponible :", MODELE in disponibles)

t0 = time.time()
try:
    r = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS, timeout=60, json={
        "model": MODELE, "max_tokens": 5, "messages": [{"role": "user", "content": "Reponds OK"}]})
    print(f"appel texte simple : HTTP {r.status_code} en {time.time() - t0:.1f} s")
except requests.Timeout:
    print(f"appel texte simple : aucune reponse en {time.time() - t0:.0f} s")

liste des modeles : HTTP 200 en 0.1 s
gemma4-26b-moe disponible : True
appel texte simple : HTTP 200 en 0.6 s


In [11]:
# 5. Les outils de liste : appartenance, et noms existants proches (1 ou 2 lettres d'ecart)
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ "

def dans_liste(valeur, champ):
    """Present si la valeur entiere, ou chacun de ses mots (noms et prenoms composes), est dans la liste."""
    if not valeur or valeur == "ILLISIBLE":
        return True                     # rien a verifier : deja une erreur avouee
    liste = LISTE[champ]
    return valeur in liste or all(m in liste for m in valeur.split())


def voisins1(mot):
    """Toutes les chaines a une operation d'ecart (suppression, inversion, remplacement, ajout)."""
    coupes = [(mot[:i], mot[i:]) for i in range(len(mot) + 1)]
    sup = [a + b[1:] for a, b in coupes if b]
    inv = [a + b[1] + b[0] + b[2:] for a, b in coupes if len(b) > 1]
    rem = [a + c + b[1:] for a, b in coupes if b for c in ALPHABET]
    aj = [a + c + b for a, b in coupes for c in ALPHABET]
    return set(sup + inv + rem + aj)


def candidats(valeur, champ, ecart_max=1, n=5):
    """Valeurs existantes les plus frequentes a 1 (ou 2) lettres d'ecart."""
    liste = LISTE[champ]
    v1 = voisins1(valeur)
    trouves = {v for v in v1 if v in liste}
    if ecart_max >= 2 and len(trouves) < n:
        for v in list(v1):
            trouves |= {w for w in voisins1(v) if w in liste}
    trouves.discard(valeur)
    return sorted(trouves, key=lambda v: -liste[v])[:n]


LU = BASE.copy()
LU["lu_n"] = LU.lu.map(normaliser)
LU["attendu"] = [REF.get((c, ch), "") for c, ch in zip(LU.cle, LU.champ)]
LU = LU[LU.attendu != ""].reset_index(drop=True)
LU["exact"] = (LU.lu_n == LU.attendu).astype(int)
LU["avoue"] = LU.lu_n.isin(["", "ILLISIBLE"]).astype(int)
LU["silencieux"] = ((LU.exact == 0) & (LU.avoue == 0)).astype(int)
LU["absent"] = [int(not dans_liste(v, c)) for v, c in zip(LU.lu_n, LU.champ)]
print(f"{len(LU)} lignes | exact {LU.exact.mean():.1%} | silencieuses {LU.silencieux.mean():.1%} | "
      f"lecture absente de la liste {LU.absent.mean():.1%}")

335 lignes | exact 83.9% | silencieuses 15.2% | lecture absente de la liste 14.0%


In [12]:
# 6. Couverture : les VRAIS noms et prenoms (Fiqual) sont-ils dans les listes ?
couv = []
for champ in COLS:
    vrais = LU[LU.champ == champ].attendu
    dedans = vrais.map(lambda v: dans_liste(v, champ))
    lo, hi = wilson(int(dedans.sum()), len(dedans))
    couv.append({"champ": champ, "N": len(dedans), "couverture_%": round(dedans.mean() * 100, 1),
                 "IC95": f"[{lo * 100:.1f} ; {hi * 100:.1f}]"})
COUVERTURE = pd.DataFrame(couv)
display(COUVERTURE)


,champ,N,couverture_%,IC95
0,NOM,168,83.9,[77.6 ; 88.7]
1,PRENOM,167,98.2,[94.9 ; 99.4]


In [13]:
# 7. Usage 1 — le signal : les lectures absentes de la liste partent en relecture
sil = LU[LU.silencieux == 1]
justes = LU[LU.exact == 1]
attrapees = int(sil.absent.sum())
fausses = int(justes.absent.sum())
bulletins_relire = LU[LU.absent == 1].cle.nunique()
restantes = int(((LU.silencieux == 1) & (LU.absent == 0)).sum())
lo, hi = wilson(attrapees, len(sil))
SIGNAL = {
    "erreurs_silencieuses": len(sil),
    "attrapees": attrapees,
    "attrapees_%": round(attrapees / max(1, len(sil)) * 100, 1),
    "attrapees_IC95": f"[{lo * 100:.1f} ; {hi * 100:.1f}]",
    "fausses_alertes_%": round(fausses / max(1, len(justes)) * 100, 1),
    "lignes_a_relire_%": round(LU.absent.mean() * 100, 1),
    "bulletins_a_relire_%": round(bulletins_relire / LU.cle.nunique() * 100, 1),
    "silencieuses_avant_%": round(LU.silencieux.mean() * 100, 1),
    "silencieuses_apres_%": round(restantes / len(LU) * 100, 1),
}
for k, v in SIGNAL.items():
    print(f"  {k:24} {v}")
print("\nLecture : 'silencieuses_apres' = erreurs qui passeraient encore sans aucun signal.")

  erreurs_silencieuses     51
  attrapees                25
  attrapees_%              49.0
  attrapees_IC95           [35.9 ; 62.3]
  fausses_alertes_%        7.8
  lignes_a_relire_%        14.0
  bulletins_a_relire_%     24.3
  silencieuses_avant_%     15.2
  silencieuses_apres_%     7.8

Lecture : 'silencieuses_apres' = erreurs qui passeraient encore sans aucun signal.


In [14]:
# 8. Usage 2 — la correction automatique (deux regles, de la plus prudente a la plus audacieuse)
def corriger(valeur, champ, regle):
    if dans_liste(valeur, champ) or len(valeur) < LONGUEUR_MIN_CORRECTION:
        return valeur
    cands = candidats(valeur, champ, ecart_max=1)
    if regle == "unique":                   # on ne corrige que s'il n'y a qu'un seul candidat
        return cands[0] if len(cands) == 1 else valeur
    return cands[0] if cands else valeur    # "frequent" : le candidat le plus frequent

CORR = []
for regle in ["unique", "frequent"]:
    apres = pd.Series([corriger(v, c, regle) for v, c in zip(LU.lu_n, LU.champ)])
    ex = (apres == LU.attendu).astype(int)
    rep, abi, p = mcnemar(LU.exact, ex)
    CORR.append({"regle": regle, "exact_avant_%": round(LU.exact.mean() * 100, 1),
                 "exact_apres_%": round(ex.mean() * 100, 1),
                 "lignes_modifiees": int((apres != LU.lu_n).sum()),
                 "reparees": rep, "abimees": abi, "McNemar_p": round(p, 4)})
    LU[f"corr_{regle}"] = apres.values
CORRECTION = pd.DataFrame(CORR)
display(CORRECTION)
print("Lecture : 'abimees' = noms bien lus que la correction a rendus faux (le risque principal).")

,regle,exact_avant_%,exact_apres_%,lignes_modifiees,reparees,abimees,McNemar_p
0,unique,83.9,83.9,4,2,2,1.0000
1,frequent,83.9,81.8,35,9,16,0.2295


Lecture : 'abimees' = noms bien lus que la correction a rendus faux (le risque principal).


In [15]:
# 9. Usage 3 - la consigne : relecture par gemma4 des seules lignes suspectes, avec des candidats
from concurrent.futures import ThreadPoolExecutor, as_completed

cache = RESULTATS / f"listes_insee_relecture{SUFFIXE}.csv"
REL = pd.read_csv(cache, dtype=str).fillna("") if cache.exists() else pd.DataFrame(columns=["cle", "champ", "lu"])
suspects = list(LU[LU.absent == 1].cle.unique())
a_faire = [c for c in suspects if c not in set(REL.cle)]
print(f"{len(suspects)} blocs suspects : {len(suspects) - len(a_faire)} deja relus, {len(a_faire)} a relire")
log.info(f"relecture : {len(a_faire)} blocs a relire, {N_PARALLELE} en parallele")


def preparer_consigne(cle):
    """Consigne de relecture d'un bloc : premiere lecture + candidats des listes Insee pour les champs suspects."""
    lignes = LU[LU.cle == cle].set_index("champ")
    lecture = ", ".join(f"{c} = {lignes.loc[c, 'lu_n'] or 'vide'}" for c in COLS if c in lignes.index)
    props = []
    for c in COLS:
        if c in lignes.index and lignes.loc[c, "absent"] == 1:
            cands = candidats(lignes.loc[c, "lu_n"], c, ecart_max=2)
            props.append(f"- {c} : {', '.join(cands) if cands else '(aucune valeur proche)'}")
    return PROMPT_RELECTURE.format(lecture=lecture, candidats="\n".join(props))


def relire_bloc(cle):
    t0 = time.time()
    res, ok = lire(Image.open(BLOCS / f"{cle}.png"), preparer_consigne(cle), etiquette=f"relecture={cle}")
    return cle, res, ok, time.time() - t0


rows, echecs_rel, temps_rel = [], [], []
t_debut, dernier_affichage = time.time(), 0.0


def sauver_rel():
    global REL, rows
    if rows:
        REL = pd.concat([REL, pd.DataFrame(rows)], ignore_index=True)
        REL.to_csv(cache, index=False)
        rows = []


try:
    with ThreadPoolExecutor(N_PARALLELE) as ex:
        futures = [ex.submit(relire_bloc, c) for c in a_faire]
        for k, f in enumerate(as_completed(futures), 1):
            cle, res, ok, s = f.result()
            temps_rel.append(s)
            if ok:
                rows += [{"cle": cle, "champ": c, "lu": res[c]} for c in COLS]
            else:
                echecs_rel.append(cle)              # non enregistre : retente a la relance
            if k % 10 == 0 or k == len(a_faire):
                sauver_rel()
            if time.time() - dernier_affichage > 30 or k == len(a_faire):
                dernier_affichage = time.time()
                ecoule = time.time() - t_debut
                msg = (f"relecture {k}/{len(a_faire)} | {ecoule / 60:.1f} min | {np.mean(temps_rel[-20:]):.1f} s/appel | "
                       f"reste ~{ecoule / k * (len(a_faire) - k) / 60:.0f} min | echecs {len(echecs_rel)}")
                print("  " + msg, flush=True)
                log.info(msg)
finally:
    sauver_rel()                                    # interruption : rien n'est perdu

STATS_RELECTURE = {"blocs_suspects": len(suspects), "appels": len(temps_rel), "echecs": len(echecs_rel),
                   "s_par_appel": float(np.mean(temps_rel)) if temps_rel else 0.0,
                   "duree_min": (time.time() - t_debut) / 60}
log.info(f"fin de relecture : {STATS_RELECTURE}")
print(f"{REL.cle.nunique()}/{len(suspects)} blocs relus | {len(echecs_rel)} echecs"
      + (" (relancer la cellule pour les retenter)" if echecs_rel else ""))

# --- Mesure : seules les lignes suspectes sont remplacees ; les autres gardent la premiere lecture ---
if echecs_rel or REL.cle.nunique() < len(suspects):
    print("ATTENTION : relecture incomplete ; les blocs non relus gardent leur premiere lecture dans la mesure")
rel = {(r.cle, r.champ): normaliser(r.lu) for r in REL.itertuples()}
LU["consigne"] = [rel.get((k, c), v) if a == 1 else v
                  for k, c, v, a in zip(LU.cle, LU.champ, LU.lu_n, LU.absent)]
ex3 = (LU.consigne == LU.attendu).astype(int)
rep, abi, p = mcnemar(LU.exact, ex3)
CONSIGNE = {"exact_avant_%": round(LU.exact.mean() * 100, 1), "exact_apres_%": round(ex3.mean() * 100, 1),
            "lignes_relues": int(LU.absent.sum()), "reparees": rep, "abimees": abi, "McNemar_p": round(p, 4)}
for k, v in CONSIGNE.items():
    print(f"  {k:16} {v}")

41 blocs suspects : 41 deja relus, 0 a relire
41/41 blocs relus | 0 echecs
  exact_avant_%    83.9
  exact_apres_%    86.6
  lignes_relues    47
  reparees         11
  abimees          2
  McNemar_p        0.0225


In [16]:
# 10. Le tableau de synthese des trois usages
def ligne(nom, lu_final, relecture_pct=0.0):
    ex = (lu_final == LU.attendu).astype(int)
    av = lu_final.isin(["", "ILLISIBLE"]).astype(int)
    si = ((ex == 0) & (av == 0)).astype(int)
    lo, hi = wilson(int(ex.sum()), len(ex))
    return {"usage": nom, "exact_%": round(ex.mean() * 100, 1), "IC95": f"[{lo * 100:.1f} ; {hi * 100:.1f}]",
            "silencieuses_%": round(si.mean() * 100, 1), "lignes_a_relire_%": relecture_pct}

base = LU.lu_n
SYNTHESE = pd.DataFrame([
    ligne("gemma4 seul (reference)", base),
    {**ligne("1. signal (relecture des absents)", base, SIGNAL["lignes_a_relire_%"]),
     "silencieuses_%": SIGNAL["silencieuses_apres_%"]},
    ligne("2a. correction, candidat unique", LU.corr_unique),
    ligne("2b. correction, candidat le plus frequent", LU.corr_frequent),
    ligne("3. consigne avec candidats", LU.consigne),
])
SYNTHESE.to_csv(RESULTATS / f"listes_insee_synthese{SUFFIXE}.csv", index=False)
with pd.option_context("display.max_columns", 10, "display.width", 200):
    display(SYNTHESE)
print("\nPour le signal, 'silencieuses' = erreurs qui passent SANS etre envoyees en relecture.")

,usage,exact_%,IC95,silencieuses_%,lignes_a_relire_%
0,gemma4 seul (reference),83.9,[79.6 ; 87.4],15.2,0.0
1,1. signal (relecture des absents),83.9,[79.6 ; 87.4],7.8,14.0
2,"2a. correction, candidat unique",83.9,[79.6 ; 87.4],15.2,0.0
3,"2b. correction, candidat le plus frequent",81.8,[77.3 ; 85.6],17.3,0.0
4,3. consigne avec candidats,86.6,[82.5 ; 89.8],12.5,0.0



Pour le signal, 'silencieuses' = erreurs qui passent SANS etre envoyees en relecture.


In [ ]:
# 11. Trace MLflow (mode complet seulement)
if TEST_RAPIDE:
    print("Mode TEST RAPIDE : rien n'est enregistre. Passer TEST_RAPIDE = False et relancer 1 a 11.")
else:
    mlflow.set_experiment("fiqual-banc-essai")
    with mlflow.start_run(run_name="listes-insee") as run:
        mlflow.set_tags({"etude": "dico-noms", "notebook": "09", "statut": "exploratoire", "modele": MODELE})
        mlflow.log_params({"bulletins": LU.cle.nunique(), "lignes": len(LU),
                           "longueur_min_correction": LONGUEUR_MIN_CORRECTION,
                           "liste_noms": URL_NOMS.split("/")[-1], "liste_prenoms": URL_PRENOMS.split("/")[-1]})
        for r in COUVERTURE.itertuples():
            mlflow.log_metric(f"couverture_{r.champ}_pct", float(r._3))
        for k, v in SIGNAL.items():
            if isinstance(v, (int, float)):
                mlflow.log_metric(f"signal_{k.replace('%', 'pct')}", float(v))
        for _, r in CORRECTION.iterrows():
            for k in ["exact_apres_%", "reparees", "abimees", "McNemar_p"]:
                mlflow.log_metric(f"correction_{r['regle']}_{k.replace('%', 'pct')}", float(r[k]))
        for k, v in CONSIGNE.items():
            mlflow.log_metric(f"consigne_{k.replace('%', 'pct')}", float(v))
        mlflow.log_artifact(str(RESULTATS / "listes_insee_synthese.csv"))
    print("essai enregistre :", run.info.run_id[:8])